# Supplementary Material 6 — Cohort Characteristics

This supplement describes the datasets used for evaluating the patient simulator, including data sources, cleaning procedures, cohort demographics, outcome distributions, labelling methodology, and inter-rater agreement.

**Datasets:**
- **AV (Dataset 1)**: 772 synthetic clinical vignettes with 5-grader triage labels
- **HV (Dataset 2)**: 394 real EHR-derived encounters with 3-annotator triage labels

Both datasets include simulated patient–doctor conversations and expert-assigned triage urgency labels used as ground truth for downstream evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path
from IPython.display import display, HTML
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters
from itertools import combinations

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 5)

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / ".git").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
AV_DIR = DATA_DIR / "av"
HV_DIR = DATA_DIR / "hv"

av = pd.read_csv(AV_DIR / "av_clean.csv")
hv = pd.read_csv(HV_DIR / "hv_clean.csv")

print(f"AV: {av.shape[0]} cases, {av.shape[1]} columns")
print(f"HV: {hv.shape[0]} encounters, {hv.shape[1]} columns")

## 1. Data Sources

| Property | AV (Dataset 1) | HV (Dataset 2) |
|---|---|---|
| **Source** | Synthetically generated clinical vignettes | De-identified real EHR encounters |
| **Generation** | LLM-generated patient scenarios with structured clinical data | Extracted from electronic health records |
| **Conversations** | Simulated patient–doctor dialogues (1 per case) | Real patient–doctor dialogues from clinical encounters |
| **Triage labels** | 5 expert graders per case, majority vote | 3 clinical annotators per case, majority vote |
| **Triage scale** | 3-level (self-care, non-urgent, urgent) | 3-level (self-care, follow-up PCP, urgent/emergency) |

## 2. AV Dataset

### 2.1 Data Cleaning & Attrition

In [ ]:
# AV Attrition table
attrition_av = pd.DataFrame([
    {"Step": 0, "Action": "Load merged_case.tsv", "N_in": 799, "Dropped": 0, "N_out": 799,
     "Sex (F/M)": "417 / 382", "Age mean (SD)": "48.8 (17.5)"},
    {"Step": 1, "Action": "Remove duplicate cases (similarity ≥ 0.90)", "N_in": 799, "Dropped": 26, "N_out": 773,
     "Sex (F/M)": "410 / 363", "Age mean (SD)": "48.8 (17.6)"},
    {"Step": 2, "Action": "Remove underage cases (age < 18)", "N_in": 773, "Dropped": 1, "N_out": 772,
     "Sex (F/M)": "410 / 362", "Age mean (SD)": "48.9 (17.6)"},
    {"Step": 3, "Action": "Merge grader votes & conversations", "N_in": 772, "Dropped": 0, "N_out": 772,
     "Sex (F/M)": "410 / 362", "Age mean (SD)": "48.9 (17.6)"},
    {"Step": 4, "Action": "Drop AI performance columns (9 cols)", "N_in": 772, "Dropped": 0, "N_out": 772,
     "Sex (F/M)": "410 / 362", "Age mean (SD)": "48.9 (17.6)"},
])
display(attrition_av.set_index("Step"))

### 2.2 Data Dictionary

In [ ]:
av_dict = pd.read_csv(AV_DIR / "data_dictionary.csv")
display(av_dict)

### 2.3 Example Case

In [ ]:
# Show a complete case from each triage level
triage_map_av = {
    "pursue self-care and monitoring": "Self-care",
    "seek non-urgent follow-up care": "Non-urgent",
    "seek immediate medical care": "Urgent",
}

example = av.iloc[0]
print(f"Case ID: {example['case']}")
print(f"Sex: {example['sex']}, Age: {example['age']}")
print(f"Chief symptom: {example['chief_symptom'][:200]}")
print(f"Chronic conditions: {example.get('chronic_conditions', 'N/A')}")
print(f"ICD code: {example['ICD9_10 code']}")
print(f"Grader triage (majority): {example['grader_triage']}")
print(f"Individual graders: {[example[f'grader_{i}_triage'] for i in range(1,6)]}")
print(f"\nPatient summary (first 500 chars):")
print(example["patient_summary"][:500])
print(f"\nConversation (first 800 chars):")
print(example["conversation"][:800])

### 2.4 Demographics

In [ ]:
# AV Demographics table
def demographics_table(df, sex_col="sex", age_col="age"):
    n = len(df)
    sex_counts = df[sex_col].value_counts()
    age = df[age_col].dropna()

    rows = [
        {"Variable": "N", "Value": str(n)},
        {"Variable": "Sex — Female", "Value": f"{sex_counts.get('Female', sex_counts.get('F', 0))} "
         f"({sex_counts.get('Female', sex_counts.get('F', 0))/n:.1%})"},
        {"Variable": "Sex — Male", "Value": f"{sex_counts.get('Male', sex_counts.get('M', 0))} "
         f"({sex_counts.get('Male', sex_counts.get('M', 0))/n:.1%})"},
        {"Variable": "Age — Mean (SD)", "Value": f"{age.mean():.1f} ({age.std():.1f})"},
        {"Variable": "Age — Median [IQR]",
         "Value": f"{age.median():.0f} [{age.quantile(0.25):.0f}–{age.quantile(0.75):.0f}]"},
        {"Variable": "Age — Range", "Value": f"{age.min():.0f}–{age.max():.0f}"},
    ]
    return pd.DataFrame(rows)

print("=== AV Demographics ===")
display(demographics_table(av))

# Age distribution by sex
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, sex in zip(axes, ["Female", "Male"]):
    subset = av[av["sex"] == sex]["age"]
    ax.hist(subset, bins=20, color="steelblue" if sex == "Male" else "coral",
            edgecolor="white", alpha=0.8)
    ax.set_title(f"{sex} (n={len(subset)})")
    ax.set_xlabel("Age")
    ax.set_ylabel("Count")
    ax.axvline(subset.median(), color="black", ls="--", lw=1,
               label=f"median={subset.median():.0f}")
    ax.legend(fontsize=9)
plt.suptitle("AV — Age distribution by sex", fontsize=13)
plt.tight_layout()
plt.show()

### 2.5 Triage Outcome Distribution

In [ ]:
# AV triage distribution
triage_order_av = ["pursue self-care and monitoring",
                   "seek non-urgent follow-up care",
                   "seek immediate medical care"]

triage_counts = av["grader_triage"].value_counts()
triage_df = pd.DataFrame({
    "Triage level": triage_order_av,
    "N": [triage_counts.get(t, 0) for t in triage_order_av],
    "%": [f"{triage_counts.get(t, 0)/len(av):.1%}" for t in triage_order_av],
})
unevaluable = triage_counts.get("unevaluable", 0)
if unevaluable:
    triage_df = pd.concat([triage_df, pd.DataFrame([{
        "Triage level": "unevaluable", "N": unevaluable,
        "%": f"{unevaluable/len(av):.1%}"}])], ignore_index=True)

print("=== AV Triage Distribution (Grader Majority Vote) ===")
display(triage_df)

### 2.6 Grader Information & Inter-Rater Agreement

Each AV case was independently rated by 5 graders (from a pool of 17) on a 3-level triage scale. The ground-truth label is the majority vote across the 5 graders.

In [ ]:
# Grader agreement levels
grader_cols = [f"grader_{i}_triage" for i in range(1, 6)]

# Count how many graders agree with majority per case
def count_majority_agree(row):
    votes = [row[c] for c in grader_cols if pd.notna(row[c])]
    if not votes:
        return 0
    majority = Counter(votes).most_common(1)[0][0]
    return sum(1 for v in votes if v == majority)

av["n_agree"] = av.apply(count_majority_agree, axis=1)

agree_df = pd.DataFrame({
    "Agreement level": ["5/5 (unanimous)", "≥4/5 (strong)", "≥3/5 (majority)"],
    "N": [(av["n_agree"] == 5).sum(),
          (av["n_agree"] >= 4).sum(),
          (av["n_agree"] >= 3).sum()],
    "%": [f"{(av['n_agree'] == 5).mean():.1%}",
          f"{(av['n_agree'] >= 4).mean():.1%}",
          f"{(av['n_agree'] >= 3).mean():.1%}"],
})
print("=== Grader Agreement Levels ===")
display(agree_df)

# Cohen's kappa (pairwise, linear weights) & Fleiss' kappa
rating_matrix = av[grader_cols].dropna()
print(f"\nCases with all 5 grader ratings: {len(rating_matrix)}")

print("\n=== Pairwise Cohen's κ (linear weights) ===")
kappas = []
for i, j in combinations(range(5), 2):
    k = cohen_kappa_score(rating_matrix.iloc[:, i], rating_matrix.iloc[:, j],
                          weights="linear")
    kappas.append(k)
    print(f"  grader_{i+1} vs grader_{j+1}: κ = {k:.3f}")
mean_k = np.mean(kappas)
print(f"  Mean pairwise κ = {mean_k:.3f}")

table, cats = aggregate_raters(rating_matrix.values)
fk = fleiss_kappa(table, method="fleiss")
print(f"\n=== Fleiss' κ (all 5 graders) = {fk:.3f} ===")

# Interpretation
for name, val in [("Mean pairwise Cohen's κ", mean_k), ("Fleiss' κ", fk)]:
    if val < 0: interp = "Poor"
    elif val < 0.21: interp = "Slight"
    elif val < 0.41: interp = "Fair"
    elif val < 0.61: interp = "Moderate"
    elif val < 0.81: interp = "Substantial"
    else: interp = "Almost perfect"
    print(f"  {name} = {val:.3f} → {interp} agreement (Landis & Koch 1977)")

### 2.7 Missingness

In [ ]:
# AV missingness table
missing_av = pd.DataFrame({
    "Column": av.columns,
    "Non-null": av.notna().sum().values,
    "Missing": av.isna().sum().values,
    "Missing %": [f"{v:.1%}" for v in av.isna().mean().values],
})
missing_av = missing_av[missing_av["Missing"] > 0].sort_values("Missing", ascending=False)
if len(missing_av) == 0:
    print("No missing values in the AV dataset.")
else:
    print(f"=== AV Columns with Missing Values ({len(missing_av)} of {len(av.columns)}) ===")
    display(missing_av.reset_index(drop=True))

## 3. HV Dataset

### 3.1 Data Cleaning & Attrition

In [ ]:
attrition_hv = pd.DataFrame([
    {"Step": 0, "Action": "Load reference_simulated_conversations.csv",
     "N_in": 560, "Dropped": 0, "N_out": 560,
     "Sex (F/M)": "—", "Age mean (SD)": "—"},
    {"Step": 1, "Action": "Inner join on encounter_id",
     "N_in": 560, "Dropped": 157, "N_out": 403,
     "Sex (F/M)": "328 / 75", "Age mean (SD)": "54.4 (18.8)"},
    {"Step": 2, "Action": "Drop rows missing annotator triage",
     "N_in": 403, "Dropped": 2, "N_out": 401,
     "Sex (F/M)": "326 / 75", "Age mean (SD)": "54.4 (18.6)"},
    {"Step": 3, "Action": "Drop duplicate encounter_ids",
     "N_in": 401, "Dropped": 7, "N_out": 394,
     "Sex (F/M)": "320 / 74", "Age mean (SD)": "54.4 (18.5)"},
    {"Step": 4, "Action": "Compute majority-vote triage label",
     "N_in": 394, "Dropped": 0, "N_out": 394,
     "Sex (F/M)": "320 / 74", "Age mean (SD)": "54.4 (18.5)"},
    {"Step": 5, "Action": "Select columns (drop AI: 2 cols)",
     "N_in": 394, "Dropped": 0, "N_out": 394,
     "Sex (F/M)": "320 / 74", "Age mean (SD)": "54.4 (18.5)"},
])
display(attrition_hv.set_index("Step"))

### 3.2 Data Dictionary

In [ ]:
hv_dict = pd.read_csv(HV_DIR / "data_dictionary.csv")
display(hv_dict)

### 3.3 Example Case

In [ ]:
example_hv = hv.iloc[0]
print(f"Encounter ID: {example_hv['encounter_id']}")
print(f"Patient ID: {example_hv['patient_id']}")
print(f"Encounter date: {example_hv['encounter_date']}")
print(f"Gender: {example_hv['gender']}, Age: {example_hv['age']}")
print(f"Chief complaint: {example_hv['chief_complaint']}")
print(f"Chief complaint category: {example_hv['chief_complaint_category']}")
print(f"Condition severity: {example_hv['condition_severity']}")
print(f"Symptom onset: {example_hv['symptom_onset_timing']}")
print(f"Encounter purpose: {example_hv['encounter_purpose']}")
print(f"Chronic conditions: {example_hv['chronic_conditions']}")
print(f"Allergies: {example_hv['allergies']}")

annot_cols = ["annotator_1_triage_recommendation",
              "annotator_2_triage_recommendation",
              "annotator_3_triage_recommendation"]
print(f"\nAnnotator triage votes: {[example_hv[c] for c in annot_cols]}")
print(f"Triage label (majority): {example_hv['triage_label']}")

print(f"\nHistory of present illness (first 500 chars):")
hpi = example_hv.get("history_of_present_illness", "N/A")
print(str(hpi)[:500] if pd.notna(hpi) else "N/A")
print(f"\nConversation (first 800 chars):")
print(str(example_hv["conversation"])[:800])

### 3.4 Demographics

In [ ]:
print("=== HV Demographics ===")
display(demographics_table(hv, sex_col="gender", age_col="age"))

# Age distribution by sex
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, sex in zip(axes, ["F", "M"]):
    subset = hv[hv["gender"] == sex]["age"]
    ax.hist(subset, bins=20, color="steelblue" if sex == "M" else "coral",
            edgecolor="white", alpha=0.8)
    ax.set_title(f"{'Female' if sex == 'F' else 'Male'} (n={len(subset)})")
    ax.set_xlabel("Age")
    ax.set_ylabel("Count")
    ax.axvline(subset.median(), color="black", ls="--", lw=1,
               label=f"median={subset.median():.0f}")
    ax.legend(fontsize=9)
plt.suptitle("HV — Age distribution by sex", fontsize=13)
plt.tight_layout()
plt.show()

### 3.5 Triage Outcome Distribution

In [ ]:
triage_order_hv = ["Self-care", "Follow up with PCP", "Urgent care / emergency"]

triage_counts_hv = hv["triage_label"].value_counts()
triage_df_hv = pd.DataFrame({
    "Triage level": triage_order_hv,
    "N": [triage_counts_hv.get(t, 0) for t in triage_order_hv],
    "%": [f"{triage_counts_hv.get(t, 0)/len(hv):.1%}" for t in triage_order_hv],
})
print("=== HV Triage Distribution (Annotator Majority Vote) ===")
display(triage_df_hv)

### 3.6 Clinical Metadata

In [ ]:
# HV-specific structured metadata
metadata_tables = {
    "Chief Complaint Category": hv["chief_complaint_category"],
    "Condition Severity": hv["condition_severity"],
    "Encounter Purpose": hv["encounter_purpose"],
    "Symptom Onset Timing": hv["symptom_onset_timing"],
}

for title, col in metadata_tables.items():
    counts = col.value_counts()
    meta_df = pd.DataFrame({
        title: counts.index,
        "N": counts.values,
        "%": [f"{v/len(hv):.1%}" for v in counts.values],
    })
    print(f"\n=== {title} ===")
    display(meta_df)

### 3.7 Annotator Information & Inter-Rater Agreement

Each HV encounter was independently rated by 3 clinical annotators on a 3-level triage scale. The ground-truth label is the majority vote across the 3 annotators.

In [ ]:
annot_cols = ["annotator_1_triage_recommendation",
              "annotator_2_triage_recommendation",
              "annotator_3_triage_recommendation"]

# Raw agreement
all_agree = ((hv[annot_cols[0]] == hv[annot_cols[1]]) &
             (hv[annot_cols[1]] == hv[annot_cols[2]]))
two_agree = ((hv[annot_cols[0]] == hv[annot_cols[1]]) |
             (hv[annot_cols[0]] == hv[annot_cols[2]]) |
             (hv[annot_cols[1]] == hv[annot_cols[2]]))

agree_df_hv = pd.DataFrame({
    "Agreement level": ["3/3 (unanimous)", "≥2/3 (majority)"],
    "N": [all_agree.sum(), two_agree.sum()],
    "%": [f"{all_agree.mean():.1%}", f"{two_agree.mean():.1%}"],
})
print("=== Annotator Agreement Levels ===")
display(agree_df_hv)

# Pairwise Cohen's kappa
print("\n=== Pairwise Cohen's κ (linear weights) ===")
rating_matrix_hv = hv[annot_cols]
kappas_hv = []
for i, j in combinations(range(3), 2):
    k = cohen_kappa_score(rating_matrix_hv.iloc[:, i], rating_matrix_hv.iloc[:, j],
                          weights="linear")
    kappas_hv.append(k)
    print(f"  annotator_{i+1} vs annotator_{j+1}: κ = {k:.3f}")
mean_k_hv = np.mean(kappas_hv)
print(f"  Mean pairwise κ = {mean_k_hv:.3f}")

# Fleiss' kappa
table_hv, cats_hv = aggregate_raters(rating_matrix_hv.values)
fk_hv = fleiss_kappa(table_hv, method="fleiss")
print(f"\n=== Fleiss' κ (all 3 annotators) = {fk_hv:.3f} ===")

for name, val in [("Mean pairwise Cohen's κ", mean_k_hv), ("Fleiss' κ", fk_hv)]:
    if val < 0: interp = "Poor"
    elif val < 0.21: interp = "Slight"
    elif val < 0.41: interp = "Fair"
    elif val < 0.61: interp = "Moderate"
    elif val < 0.81: interp = "Substantial"
    else: interp = "Almost perfect"
    print(f"  {name} = {val:.3f} → {interp} agreement (Landis & Koch 1977)")

# Pairwise raw agreement
print("\n=== Pairwise Raw Agreement ===")
for i, j in combinations(range(3), 2):
    agree = (rating_matrix_hv.iloc[:, i] == rating_matrix_hv.iloc[:, j]).sum()
    print(f"  annotator_{i+1} vs annotator_{j+1}: {agree}/{len(hv)} ({agree/len(hv):.1%})")

### 3.8 Missingness

In [ ]:
missing_hv = pd.DataFrame({
    "Column": hv.columns,
    "Non-null": hv.notna().sum().values,
    "Missing": hv.isna().sum().values,
    "Missing %": [f"{v:.1%}" for v in hv.isna().mean().values],
})
missing_hv = missing_hv[missing_hv["Missing"] > 0].sort_values("Missing", ascending=False)
if len(missing_hv) == 0:
    print("No missing values in the HV dataset.")
else:
    print(f"=== HV Columns with Missing Values ({len(missing_hv)} of {len(hv.columns)}) ===")
    display(missing_hv.reset_index(drop=True))

## 4. Merged Cohort

For downstream experiments (fairness, performance estimation), the AV and HV datasets are combined into a single evaluation cohort. Triage labels are harmonised to a common 3-level scale:

| AV label | HV label | Harmonised label |
|---|---|---|
| pursue self-care and monitoring | Self-care | self-care |
| seek non-urgent follow-up care | Follow up with PCP | non-urgent |
| seek immediate medical care | Urgent care / emergency | urgent |

In [ ]:
# Harmonise triage labels
av_harmonised = av[["case", "sex", "age", "grader_triage"]].copy()
av_harmonised["dataset"] = "AV"
av_harmonised.rename(columns={"case": "id", "sex": "sex", "grader_triage": "triage_raw"}, inplace=True)
triage_map_av = {
    "pursue self-care and monitoring": "self-care",
    "seek non-urgent follow-up care": "non-urgent",
    "seek immediate medical care": "urgent",
}
av_harmonised["triage"] = av_harmonised["triage_raw"].map(triage_map_av)
av_harmonised["sex"] = av_harmonised["sex"].map({"Female": "F", "Male": "M"})

hv_harmonised = hv[["encounter_id", "gender", "age", "triage_label"]].copy()
hv_harmonised["dataset"] = "HV"
hv_harmonised.rename(columns={"encounter_id": "id", "gender": "sex", "triage_label": "triage_raw"}, inplace=True)
triage_map_hv = {
    "Self-care": "self-care",
    "Follow up with PCP": "non-urgent",
    "Urgent care / emergency": "urgent",
}
hv_harmonised["triage"] = hv_harmonised["triage_raw"].map(triage_map_hv)

merged = pd.concat([av_harmonised, hv_harmonised], ignore_index=True)
merged = merged[merged["triage"].notna()]
print(f"Merged cohort: {len(merged)} cases ({(merged['dataset']=='AV').sum()} AV + {(merged['dataset']=='HV').sum()} HV)")

### 4.1 Table 1 — Cohort Characteristics

In [ ]:
def table1_row(df, label=""):
    n = len(df)
    sex = df["sex"].value_counts()
    age = df["age"].dropna()
    triage = df["triage"].value_counts()
    return {
        "": label,
        "N": n,
        "Female, n (%)": f"{sex.get('F', 0)} ({sex.get('F', 0)/n:.1%})",
        "Male, n (%)": f"{sex.get('M', 0)} ({sex.get('M', 0)/n:.1%})",
        "Age, mean (SD)": f"{age.mean():.1f} ({age.std():.1f})",
        "Age, median [IQR]": f"{age.median():.0f} [{age.quantile(0.25):.0f}–{age.quantile(0.75):.0f}]",
        "Self-care, n (%)": f"{triage.get('self-care', 0)} ({triage.get('self-care', 0)/n:.1%})",
        "Non-urgent, n (%)": f"{triage.get('non-urgent', 0)} ({triage.get('non-urgent', 0)/n:.1%})",
        "Urgent, n (%)": f"{triage.get('urgent', 0)} ({triage.get('urgent', 0)/n:.1%})",
    }

table1 = pd.DataFrame([
    table1_row(merged[merged["dataset"] == "AV"], "AV"),
    table1_row(merged[merged["dataset"] == "HV"], "HV"),
    table1_row(merged, "Combined"),
]).set_index("")

print("=== Table 1: Cohort Characteristics ===")
display(table1.T)

### 4.2 Comparison of Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Age
for ds, color in [("AV", "steelblue"), ("HV", "coral")]:
    subset = merged[merged["dataset"] == ds]["age"]
    axes[0].hist(subset, bins=20, color=color, alpha=0.6, edgecolor="white", label=ds)
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Count")
axes[0].set_title("Age distribution")
axes[0].legend()

# Sex
sex_ct = pd.crosstab(merged["dataset"], merged["sex"], normalize="index") * 100
sex_ct[["F", "M"]].plot.bar(ax=axes[1], color=["coral", "steelblue"], edgecolor="none")
axes[1].set_ylabel("Percentage")
axes[1].set_title("Sex distribution")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(["Female", "Male"])

# Triage
triage_ct = pd.crosstab(merged["dataset"], merged["triage"], normalize="index") * 100
triage_ct = triage_ct.reindex(columns=["self-care", "non-urgent", "urgent"])
triage_ct.plot.bar(ax=axes[2], color=["#2ecc71", "#f39c12", "#e74c3c"], edgecolor="none")
axes[2].set_ylabel("Percentage")
axes[2].set_title("Triage distribution")
axes[2].tick_params(axis="x", rotation=0)
axes[2].legend(["Self-care", "Non-urgent", "Urgent"])

plt.suptitle("AV vs HV — Distribution Comparison", fontsize=13)
plt.tight_layout()
plt.show()

### 4.3 Inter-Rater Agreement Summary

In [ ]:
# Summary table (values will be populated when cells above run)
print("=== Inter-Rater Agreement Summary ===")
print("(Run cells above to populate actual values)\n")
agreement_summary = pd.DataFrame({
    "Dataset": ["AV", "HV"],
    "N raters": [5, 3],
    "Rating scale": ["3-level triage", "3-level triage"],
    "Unanimous agreement": ["— (run cells above)", "— (run cells above)"],
    "Mean pairwise Cohen's κ": ["— (run cells above)", "— (run cells above)"],
    "Fleiss' κ": ["— (run cells above)", "— (run cells above)"],
})
display(agreement_summary.set_index("Dataset"))

print("\nNote: Cohen's κ computed with linear weights (ordinal scale).")
print("Interpretation: Landis & Koch (1977) — <0.20 slight, 0.21–0.40 fair,")
print("0.41–0.60 moderate, 0.61–0.80 substantial, >0.80 almost perfect.")